In [ ]:
# this is a documentation of the file CAGR2.py


import sys
from pathlib import Path
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import matplotlib.colors as mcolors
# Add the parent directory to sys.path
sys.path.append(str(Path(__file__).resolve().parent.parent / 'EIAfunctions'))
from func_data_upload_OECD_salaries import data_upload_OECD_salaries
from func_plot_L import plot_matrix_columns
from func_clc_L import clc_L
from func_safe_divide import safe_divide, safe_divide_vector
from func_plot_CAGR1 import plot_CAGR1_sum_sectors


## Benchmarking report: economic impact analysis (EIA) - Canada ICT sector, 2011-2020

The Information and Communication Technology (ICT) sector is a fast-growing sector.
In this report, we employ economic impact analysis to quantify how growth in the ICT sector impacts other sectors of the economy.
The focus of this report is Canada, in comparison with the other G7 countries: France, Germany, Italy, Japan, the United Kingdom, and the United States.

In the following cell we upload data from OECD tables and save it in 3 dataframes: dfoutput, dfGDP and dfEmployment

Employment data is "Compensation of employees" in the SUT OECD table.

link to OECD data:



In [ ]:
#Fig 1: CAGR data manipulation
def clc_cagr(dfoutput, first_year, last_year, value_column):
    # pivoting is a great idea there's no groupby. there's just taking first_year and last_year, then pivoting to ahve each row isolate an equation, then we manipulate numbers in each row
    df_filtered = dfoutput[dfoutput['year'].isin([str(first_year), str(last_year)])]
    pivot_df = df_filtered.pivot_table(
        index=['country', 'sector'],
        columns='year',
        values=value_column
    ).reset_index()
    #np.where is actually an if statement np.where(condition, value_if_true, value_if_false)
    pivot_df['CAGR'] = np.where(
        (pivot_df[str(first_year)] != 0) & (pivot_df[str(first_year)].notna()) & (pivot_df[str(last_year)].notna()),
        (pivot_df[str(last_year)] / pivot_df[str(first_year)]) ** (1 / (int(last_year) - int(first_year))) - 1,
        np.nan
    )
    #plotting - take all the ICT sectors and average CAGR, then plot.
    ICT_cagr = (
        pivot_df[pivot_df['sector'].isin(ICTsectors)]   # Filter for ICT sectors
        .groupby('country')['CAGR']                     # group by country
        .mean()                                         # calculate mean CAGR for each country                   
        .sort_values(ascending=False)
    )
    return ICT_cagr

# fig 1: plot CAGR
def plot_cagr(ICT_cagr, title):
        # Rename for plotting
    ICT_cagr.index = [country_map[c] for c in ICT_cagr.index]

    # Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    x = np.arange(len(ICT_cagr))
    bars = ax.bar(x, ICT_cagr.values, color=[
        'green' if country_map.get(code, code) == 'Canada' else 'blue' for code in ICT_cagr.index
    ])

    # Add labels
    for i, bar in enumerate(bars):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, height + 0.002,
                f'{height*100:.1f}%', ha='center', va='bottom', fontsize=9)

    # Formatting
    ax.set_xticks(x)
    ax.set_xticklabels([country_map.get(code, code) for code in ICT_cagr.index], rotation=45, ha='right')
    ax.set_ylabel('Average CAGR [%]')
    ax.set_title(title)

    plt.tight_layout()
    plt.show()


# Fig 2: output share data manipulation
def get_share(dfoutput, first_year, last_year, ICTsectors, value_column):
    #the output row from OECD is the output needed according to Tanveer - it includes import and HFCE etc. I checked.
    df_filtered2 = dfoutput.pivot_table(
    index=['country', 'sector'],
    columns='year',
    values=value_column
    ).reset_index()

    yearly_output = dfoutput.groupby(['country', 'year'])[value_column].sum().reset_index()
    yearly_output = yearly_output.rename(columns={value_column: f'total yearly {value_column}'})

    for year in range(int(first_year), int(last_year) + 1):
        # merge yearly_output with df_filtered2 to get total output for each country and year
        df_filtered2 = df_filtered2.merge(
            yearly_output[yearly_output['year'] == str(year)][['country', f'total yearly {value_column}']],
            on='country',
            how='left'
        )
        df_filtered2 = df_filtered2.rename(columns={f'total yearly {value_column}': f'total yearly {value_column} {year}'})

        df_filtered2[f'{value_column} share {year}'] = df_filtered2[str(year)] / df_filtered2[f'total yearly {value_column} {year}']

    # Define the years
    years = list(range(int(first_year), int(last_year) + 1))

    # Build the list of output share columns
    share_cols = [f'{value_column} share {year}' for year in years]

    # Slice the DataFrame
    shares = df_filtered2[['country', 'sector'] + share_cols].copy()

    # Calculate the average share across the selected years
    shares[f'average_{value_column}_share'] = shares[share_cols].mean(axis=1)

    ICT_shares = shares[shares['sector'].isin(ICTsectors)][['country', 'sector', f'average_{value_column}_share']].copy()

    return shares, ICT_shares

# plot fig 2: output share
def plot_share(ICT_shares, title,value_column):
    # Group by country and sort
    country_avg = ICT_shares.groupby('country')[f'average_{value_column}_share'].mean().sort_values(ascending=False)

    # Define colors
    colors = ['green' if country == 'CAN' else 'blue' for country in country_avg.index]

    # Plot
    plt.figure(figsize=(10, 6))
    bars = plt.bar(country_avg.index, country_avg.values, color=colors)

    # Adjust y-axis limit for label space
    max_height = country_avg.max()
    plt.ylim(0, max_height * 1.15)

    # Add percentage labels
    for bar in bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            height + max_height * 0.015,
            f'{height * 100:.1f}%',
            ha='center',
            va='bottom',
            fontsize=9
        )

    plt.ylabel(f'Average ICT {value_column} Share (%)')
    plt.title(title)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

# fig2B: plot stacked output share
#def plot_stacked_output_shares(output_shares, ICT_factors, title):
def plot_stacked_shares(shares, ICT_factors, title, value_column):

    # Invert dictionary ICT_factors
    sector_to_category = {}
    for category, sectors in ICT_factors.items():
        if isinstance(sectors, list):
            for sector in sectors:
                sector_to_category[sector] = category
        else:
            sector_to_category[sectors] = category

    # Filter for ICT sectors
    ICTsectors = list(sector_to_category.keys())
    ICT_shares = shares.loc[shares['sector'].isin(ICTsectors), ['country', 'sector', f'average_{value_column}_share']].copy()

    # Map to ICT category
    ICT_shares['ICT_category'] = ICT_shares['sector'].map(sector_to_category)

    # Group by country and ICT_category, sum average_output_share
    grouped = ICT_shares.groupby(['country', 'ICT_category'])[f'average_{value_column}_share'].sum().unstack(fill_value=0)
                                                                                            # country and ICT_category are the index. unstack will create columns for each ICT_category
                                                                                            # fill_value=0 will fill NaN with 0
    # Reorder columns for consistent stacking: bottom to top
    desired_order = ['ICT - Manufacturing', 'ICT - Wholesaling', 'ICT - Software and computer services', 'ICT - Communications services']

    # Sum across ICT categories to get total ICT share per country, then sort descending
    grouped['total'] = grouped.sum(axis=1)
    grouped = grouped.sort_values('total', ascending=False).drop(columns='total')

    # Now `countries` is updated to match the new order
    countries = grouped.index.tolist()

    # Continue with plotting as before
    colors = ['#4CAF50', '#2196F3', '#FFC107', '#9C27B0']  # distinct colors
    bottom = np.zeros(len(countries))
    plt.figure(figsize=(10, 6))

    for idx, category in enumerate(desired_order):
        values = grouped[category].values
        #bars = plt.bar(countries, values, bottom=bottom, color=colors[idx], label=category)
        bars = plt.bar(countries, values * 100, bottom=bottom * 100, color=colors[idx], label=category)
        bottom += values
        
    # Add % labels on top
    for i, total in enumerate(bottom):
        plt.text(i, total * 100 + 0.2, f"{total * 100:.1f}%", ha='center', va='bottom', fontsize=9)

    #plt.ylabel('Average ICT Output Share')
    plt.ylabel(f'Average ICT {value_column} Share (%)')
    plt.title(title)
    plt.xticks(rotation=45, ha='right')
    plt.legend(title="ICT Category", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()


def plot_share_compare_frist_last_year(shares, first_year, last_year, value_column, title):

    # Invert dictionary ICT_factors
    sector_to_category = {}
    for category, sectors in ICT_factors.items():
        if isinstance(sectors, list):
            for sector in sectors:
                sector_to_category[sector] = category
        else:
            sector_to_category[sectors] = category

    # Filter for ICT sectors
    ICTsectors = list(sector_to_category.keys())
    ICT_shares_first_year = shares.loc[shares['sector'].isin(ICTsectors), ['country', 'sector', f'GDP share {first_year}']].copy()
    ICT_shares_last_year = shares.loc[shares['sector'].isin(ICTsectors), ['country', 'sector', f'GDP share {last_year}']].copy()
    # Map to ICT category
    ICT_shares_first_year['ICT_category'] = ICT_shares_first_year['sector'].map(sector_to_category)
    ICT_shares_last_year['ICT_category'] = ICT_shares_last_year['sector'].map(sector_to_category)

    # Group by country and ICT_category, sum average_output_share
    grouped_first_year = ICT_shares_first_year.groupby(['country', 'ICT_category'])[f'GDP share {first_year}'].sum().unstack(fill_value=0)
    grouped_last_year = ICT_shares_last_year.groupby(['country', 'ICT_category'])[f'GDP share {last_year}'].sum().unstack(fill_value=0)

                                                                                            # country and ICT_category are the index. unstack will create columns for each ICT_category
                                                                                            # fill_value=0 will fill NaN with 0
    # Reorder columns for consistent stacking: bottom to top
    desired_order = ['ICT - Manufacturing', 'ICT - Wholesaling', 'ICT - Software and computer services', 'ICT - Communications services']

    # Sum across ICT categories to get total ICT share per country, then sort descending
    grouped_first_year['total'] = grouped_first_year.sum(axis=1)
    grouped_last_year['total'] = grouped_last_year.sum(axis=1)
    grouped_first_year = grouped_first_year.sort_values('total', ascending=False).drop(columns='total')
    grouped_last_year = grouped_last_year.sort_values('total', ascending=False).drop(columns='total')

    # Now `countries` is updated to match the new order
    countries = grouped_last_year.index.tolist()

    # Setup
    countries = grouped_first_year.index.tolist()
    n_countries = len(countries)
    x = np.arange(n_countries)  # X positions for the bars
    bar_width = 0.35

    # Colors
    base_colors = ['#4CAF50', '#2196F3', '#FFC107', '#9C27B0']  # vivid for last_year

    # Create faded colors for first_year
    def fade_color(hex_color, blend=0.4):
        rgb = np.array(mcolors.to_rgb(hex_color))
        white = np.ones_like(rgb)
        faded_rgb = rgb * (1 - blend) + white * blend
        return faded_rgb

    faded_colors = [fade_color(c) for c in base_colors]

    # Plotting
    fig, ax = plt.subplots(figsize=(12, 6))
    bottom_first = np.zeros(n_countries)
    bottom_last = np.zeros(n_countries)

    for idx, category in enumerate(desired_order):
        values_first = grouped_first_year[category].values
        values_last = grouped_last_year[category].values

        ax.bar(x - bar_width / 2, values_first * 100, bottom=bottom_first * 100,
            color=faded_colors[idx], width=bar_width, label=f"{category} ({first_year})" if idx == 0 else "", alpha=0.8)

        ax.bar(x + bar_width / 2, values_last * 100, bottom=bottom_last * 100,
            color=base_colors[idx], width=bar_width, label=f"{category} ({last_year})" if idx == 0 else "")

        bottom_first += values_first
        bottom_last += values_last

    # Add % labels above bars for total (optional)
    for i in range(n_countries):
        ax.text(x[i] - bar_width / 2, bottom_first[i] * 100 + 1, f"{bottom_first[i] * 100:.1f}%", ha='center', fontsize=8)
        ax.text(x[i] + bar_width / 2, bottom_last[i] * 100 + 1, f"{bottom_last[i] * 100:.1f}%", ha='center', fontsize=8)

    # Final plot setup
    ax.set_ylabel(f'Average ICT {value_column} Share (%)')
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(countries, rotation=45, ha='right')
    # Add more space above the highest bar
    max_height = max(np.max(bottom_first), np.max(bottom_last)) * 100
    ax.set_ylim(top=max_height * 1.1)  # 10% extra space above tallest bar

    # Custom legend (merged by category)
    custom_legend = [Patch(color=faded_colors[i], label=f"{cat} ({first_year})") for i, cat in enumerate(desired_order)]
    custom_legend += [Patch(color=base_colors[i], label=f"{cat} ({last_year})") for i, cat in enumerate(desired_order)]
    ax.legend(handles=custom_legend, bbox_to_anchor=(1.05, 1), loc='upper left', title="ICT Category")

    plt.tight_layout()
    plt.show()





##################################################             old functions               ######################################################

def multipliers2prediction(s2s_mo, fdf_year2, column_name):
    predicted_output_year2_np  = np.round(s2s_mo.to_numpy() @ fdf_year2.values.reshape(-1, 1), 1)
    
    predicted_output_year2 = pd.DataFrame(predicted_output_year2_np, index=s2s_mo.index, columns=[column_name])
    
    return predicted_output_year2


def plot_real_vs_predicted(output_real, output_pred, 
                           income_real, income_pred, 
                           gdp_real, gdp_pred, 
                           year1, year2, title):
    fig, axes = plt.subplots(3, 1, figsize=(6,8), sharex=True)
    
    fig.suptitle(title, fontsize=16)

    # Panel 1: Output
    axes[0].plot(output_real.index, output_real, label='Real Output', color='purple', marker='o')
    axes[0].plot(output_pred.index, output_pred, label='Predicted Output', color='red', marker='o')
    axes[0].set_title(f'Output {year2} Based on {year1}')
    axes[0].set_xlabel('Sectors')
    axes[0].set_ylabel('Million USD')
    axes[0].legend()

    # Panel 2: Income
    axes[1].plot(income_real.index, income_real, label='Real Income', color='purple', marker='o')
    axes[1].plot(income_pred.index, income_pred, label='Predicted Income', color='red', marker='o')
    axes[1].set_title(f'Income {year2} Based on {year1}')
    axes[1].set_xlabel('Sectors')
    axes[1].set_ylabel('Million USD')
    axes[1].legend()

    # Panel 3: GDP
    axes[2].plot(gdp_real.index, gdp_real, label='Real GDP', color='purple', marker='o')
    axes[2].plot(gdp_pred.index, gdp_pred, label='Predicted GDP', color='red', marker='o')
    axes[2].set_title(f'GDP {year2} Based on {year1}')
    axes[2].set_xlabel('Sectors')
    axes[2].set_ylabel('Million USD')
    axes[2].legend()
    for ax in axes:
        ax.tick_params(axis='x', rotation=45)
        ax.grid(True)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()


In [ ]:

 
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@                    main                  @@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
start_time = time.time()
print("working directory of CARG2.py is: ",os.getcwd())  # Print the current working directory

table_type = 'TTL' #or'DOM'   
OECD_path = "../Data/" # windows style: r".\\"
if table_type == 'DOM':
    output_filename = '/mnt/c/NavitComputer24/2024_NES/Economics/Textbook_EIA/OECD_salaries/EIA_matrices.xlsx'
elif table_type == 'TTL':
    output_filename = '/mnt/c/NavitComputer24/2024_NES/Economics/Textbook_EIA/OECD_salaries/EIA_TTL_matrices.xlsx'

first_year = '2011'
last_year = '2020'
year_range = [str(year) for year in range(int(first_year), int(last_year) + 1)]
report_title = f'ICT sectors, {last_year}'
ICT_factors = {'ICT - Manufacturing': 'C26',
                'ICT - Wholesaling': 'G',
                'ICT - Software and computer services': ['J58T60', 'J62_63', 'M'],  
                'ICT - Communications services': 'J61'}
ICTsectors = ['C26', 'G', 'J58T60', 'J62_63', 'M', 'J61']

country_names = ['Canada', 'The United States', 'Great Britain', 'France', 'Germany', 'Italiy', 'Japan']
countries = ['CAN', 'USA', 'GBR', 'FRA', 'DEU', 'ITA', 'JPN'] # 'CHN' is not available in OECD, but it is in OECDadditional
country_map = dict(zip(countries, country_names))

currency_exchange_type = 'EXCH' #'EXCH' or 'PPP'


# 1. Get IO=II, X, GDP, from OECD, compensation of employees, more GDP and II from OECDadditional as well as taxes, incomegross surplus etc.
##########################################################################################################################################   
dftemp = pd.DataFrame()
dfoutput = pd.DataFrame() # this will hold output by country, year, sector, output
dfGDP = pd.DataFrame() # this will hold the GDP by country, year, sector, GDP
outputcagr_by_country = {}
for country in countries:
    output_by_year = {}
    for year in year_range:
        PPP_or_exch, OECD, simple_II_labels, OECDadditional, sector_description =  data_upload_OECD_salaries(year, currency_exchange_type, table_type, country)
        # the following is calculated twice: in data_upload_OECD_salaries and here. I want to leave it here, but I also need it there - do I??
        #II = OECD.loc[simple_II_labels, simple_II_labels]
        #household_expenditure = OECD.loc[simple_II_labels, 'HFCE']
        #final_demand_columns = ['HFCE',	'NPISH',	'GGFC',	'GFCF',	'INVNT',	'CONS_NONRES', 'EXPO'] # 'IMPO', 'DPABR', 
        #other_final_demand = OECD.loc[simple_II_labels, final_demand_columns[1:]] #exluding HFCE - household expenditure
        GDP         = OECD.loc['VALU', simple_II_labels]
        output      = OECD.loc['OUTPUT', simple_II_labels]
        
        dftemp = output.reset_index()
        dftemp.columns = ['sector', 'output']
        dftemp['country'] = country
        dftemp['year'] = year
        dftemp = dftemp[['country', 'year', 'sector', 'output']]
        dfoutput = pd.concat([dfoutput, dftemp], ignore_index=True)

        dftempg = GDP.reset_index()
        dftempg.columns = ['sector', 'GDP']
        dftempg['country'] = country
        dftempg['year'] = year
        dftempg = dftempg[['country', 'year', 'sector', 'GDP']]
        dfGDP = pd.concat([dfGDP, dftempg], ignore_index=True)
    



In [ ]:

print(f'Fig 1: ICT Sector Revenue Compound Annual Growth Rate (CAGR) ({first_year}-{last_year})')

if 0:
    # fig 1: output CAGR 
    ICT_cagr = clc_cagr(dfoutput, first_year, last_year,'output')
    # fig1: plot output CAGR
    plot_cagr(ICT_cagr, f'Average Output CAGR for ICT sectors ({first_year}–{last_year})')


    print(f'Fig 2: Average ICT Sector Share in Total National Output ({first_year}-{last_year})')
    # fig 2: average ICT sector share in total output 2010-2020
    # data manipulation for figure 2: output share of ICT sectors
    output_shares, ICT_output_shares = get_share(dfoutput, first_year, last_year, ICTsectors,'output')
    # TODO: I'm not sure if ICT_output_shares should be in data manipulation or plotting
    plot_share(ICT_output_shares, f'Average ICT Output Share by Country, {first_year}-{last_year}','output')
    #the above is average of average - average over the 6 ICT sectorsa as well as over the years

    # fig2B: stacked output share
    #this is the average of each category (factor) - stacked. 
    plot_stacked_shares(output_shares, ICT_factors,f'Stacked Average ICT Output Share by Country, {first_year}-{last_year}','output')



# graphs 1 and 2 for GDP
if 0:
    # fig 1: output CAGR 
    ICT_GDP_cagr = clc_cagr(dfGDP, first_year, last_year,'GDP') 
    # fig1: plot output CAGR
    plot_cagr(ICT_GDP_cagr, f'Average GDP CAGR for ICT sectors ({first_year}–{last_year})')

    # fig 2: average ICT sector share in GDP 2011-2020
    GDP_shares, ICT_GDP_shares = get_share(dfGDP, first_year, last_year, ICTsectors,'GDP')
    plot_share(ICT_GDP_shares, f'Average ICT GDP Share by Country, {first_year}-{last_year}','GDP')

    # fig2B: stacked output share
    #this is the average of each category (factor) - stacked. 
    plot_stacked_shares(GDP_shares, ICT_factors,f'Stacked Average ICT GDP Share by Country, {first_year}-{last_year}','GDP')

#GDP share stacked, not average but comparison between 2011 and 2020
if 1:
    GDP_shares, ICT_GDP_shares = get_share(dfGDP, first_year, last_year, ICTsectors,'GDP')
    plot_share_compare_frist_last_year(GDP_shares, first_year, last_year, 'GDP', f'ICT GDP {first_year} and {last_year} Share by Country')




print('graphs 1 and 2 are done')
# from now on it is not compiling